## Carga y Extracción de Datos con Web Scraping

El **Web Scraping** es la técnica programática para extraer información no estructurada o semiestructurada directamente del código HTML de páginas web y transformarla en conjuntos de datos estructurados (tablas, CSV, DataFrames).

### Elementos Fundamentales:
- **HTML (HyperText Markup Language)**: Lenguaje de marcado de la web compuesto por etiquetas (`<div>`, `<p>`, `<table>`, `<a>`, `<span>`).
- **DOM (Document Object Model)**: Estructura de árbol jerárquica que representa el documento HTML en memoria.
- **Selectores CSS / Clases / IDs**: Métodos para localizar elementos precisos en el árbol DOM.
- **Buenas Prácticas y Ética**:
  - Consultar siempre el archivo `robots.txt` del sitio (ej. `sitio.com/robots.txt`).
  - Implementar tiempos de espera (rate limiting) para evitar sobrecargar servidores.
  - Respetar los Términos de Servicio y la privacidad de los datos.

En este ejercicio extraeremos datos de catálogo del portal educativo **Books to Scrape** (`http://books.toscrape.com`), un entorno web creado expresamente para prácticas de scraping.

In [ ]:
%pip install requests beautifulsoup4 pandas lxml

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import re

print("Librerías importadas exitosamente.")

## 1. Descarga del Documento HTML con `requests`

In [ ]:
# URL objetivo: Books to Scrape
URL = "http://books.toscrape.com/"
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
}

try:
    response = requests.get(URL, headers=headers, timeout=10)
    response.raise_for_status()
    html_content = response.text
    print(f"Página descargada exitosamente. Código HTTP: {response.status_code}")
except Exception as e:
    print(f"Aviso de conexión: ({e}). Se utiliza contenido HTML de respaldo para garantizar la ejecución.")
    html_content = '''
    <!DOCTYPE html>
    <html>
      <head><title>All products | Books to Scrape - Sandbox</title></head>
      <body>
        <div class="col-sm-8 col-md-9">
          <ol class="row">
            <li class="col-xs-6 col-sm-4 col-md-3 col-lg-3">
              <article class="product_pod">
                <div class="image_container"><a href="#"><img src="media/cache/2c/da/2cdad67c44b002e7ead0cc35693c0e8b.jpg" alt="A Light in the Attic" class="thumbnail"></a></div>
                <p class="star-rating Three"><i class="icon-star"></i><i class="icon-star"></i><i class="icon-star"></i></p>
                <h3><a href="#" title="A Light in the Attic">A Light in the Attic</a></h3>
                <div class="product_price">
                  <p class="price_color">£51.77</p>
                  <p class="instock availability"><i class="icon-ok"></i> In stock</p>
                </div>
              </article>
            </li>
            <li class="col-xs-6 col-sm-4 col-md-3 col-lg-3">
              <article class="product_pod">
                <div class="image_container"><a href="#"><img src="media/cache/26/0c/260c6ae16bce31c8f8c95daddd9f4a1c.jpg" alt="Tipping the Velvet" class="thumbnail"></a></div>
                <p class="star-rating One"><i class="icon-star"></i></p>
                <h3><a href="#" title="Tipping the Velvet">Tipping the Velvet</a></h3>
                <div class="product_price">
                  <p class="price_color">£53.74</p>
                  <p class="instock availability"><i class="icon-ok"></i> In stock</p>
                </div>
              </article>
            </li>
            <li class="col-xs-6 col-sm-4 col-md-3 col-lg-3">
              <article class="product_pod">
                <div class="image_container"><a href="#"><img src="media/cache/3e/ef/3eef99c9d9e431b3e9f9781476a9b4f9.jpg" alt="Soumission" class="thumbnail"></a></div>
                <p class="star-rating One"><i class="icon-star"></i></p>
                <h3><a href="#" title="Soumission">Soumission</a></h3>
                <div class="product_price">
                  <p class="price_color">£50.10</p>
                  <p class="instock availability"><i class="icon-ok"></i> In stock</p>
                </div>
              </article>
            </li>
            <li class="col-xs-6 col-sm-4 col-md-3 col-lg-3">
              <article class="product_pod">
                <div class="image_container"><a href="#"><img src="media/cache/32/e1/32e1047d9d623b79eb74b26ccace2779.jpg" alt="Sharp Objects" class="thumbnail"></a></div>
                <p class="star-rating Four"><i class="icon-star"></i><i class="icon-star"></i><i class="icon-star"></i><i class="icon-star"></i></p>
                <h3><a href="#" title="Sharp Objects">Sharp Objects</a></h3>
                <div class="product_price">
                  <p class="price_color">£47.82</p>
                  <p class="instock availability"><i class="icon-ok"></i> In stock</p>
                </div>
              </article>
            </li>
          </ol>
        </div>
      </body>
    </html>
    '''

# Parsear el HTML usando BeautifulSoup con el analizador 'html.parser'
soup = BeautifulSoup(html_content, 'html.parser')
print(f"Título del sitio web: {soup.title.string.strip() if soup.title else 'Books to Scrape'}")

## 2. Inspección y Extracción de Elementos con BeautifulSoup

Cada libro en la tienda está encapsulado dentro de una etiqueta `<article class="product_pod">`. De cada elemento extraeremos:
1. **Título**: Atributo `title` en el enlace `<a>` dentro de `<h3>`.
2. **Precio**: Texto dentro de `<p class="price_color">`.
3. **Calificación**: Clase secundaria de `<p class="star-rating ...">` (One, Two, Three, Four, Five).
4. **Disponibilidad**: Texto dentro de `<p class="availability">`.

In [ ]:
# Mapeo de texto de calificación a valor numérico
rating_map = {
    'One': 1,
    'Two': 2,
    'Three': 3,
    'Four': 4,
    'Five': 5
}

# Localizar todos los artículos de productos
articulos = soup.find_all('article', class_='product_pod')
print(f"Cantidad de libros detectados en la página: {len(articulos)}")

libros_lista = []

for item in articulos:
    # 1. Título
    h3_tag = item.find('h3')
    a_tag = h3_tag.find('a') if h3_tag else None
    titulo = a_tag.get('title', a_tag.text.strip()) if a_tag else "Sin título"
    
    # 2. Precio
    precio_tag = item.find('p', class_='price_color')
    precio_raw = precio_tag.text.strip() if precio_tag else "£0.00"
    
    # 3. Calificación en estrellas
    rating_tag = item.find('p', class_=re.compile(r'star-rating'))
    estrellas = 0
    if rating_tag:
        for clase in rating_tag.get('class', []):
            if clase in rating_map:
                estrellas = rating_map[clase]
                break
                
    # 4. Disponibilidad
    disp_tag = item.find('p', class_='availability')
    disponibilidad = disp_tag.text.strip() if disp_tag else "Desconocido"
    
    libros_lista.append({
        'titulo': titulo,
        'precio_crudo': precio_raw,
        'calificacion': estrellas,
        'disponibilidad': disponibilidad
    })

print(f"Se extrajeron {len(libros_lista)} libros con éxito. Primeros 2:")
libros_lista[:2]

## 3. Conversión a Pandas y Limpieza de Datos

In [ ]:
# Crear DataFrame a partir de los datos scrapeados
df_libros = pd.DataFrame(libros_lista)

# Limpiar precios: eliminar símbolo £ y convertir a valor numérico flotante
df_libros['precio'] = df_libros['precio_crudo'].str.replace('£', '', regex=False).str.replace('$', '', regex=False).str.strip().astype(float)

# Limpiar disponibilidad en un booleano (True si está en stock)
df_libros['en_stock'] = df_libros['disponibilidad'].str.contains('In stock', case=False)

# Eliminar columnas crudas temporales
df_libros = df_libros.drop(columns=['precio_crudo', 'disponibilidad'])

print(f"Dimensiones del DataFrame de libros: {df_libros.shape}")
df_libros.head()

In [ ]:
# Estadísticas descriptivas de los libros scrapeados
df_libros.describe()

In [ ]:
# Precio promedio según la calificación de estrellas
promedios_por_rating = df_libros.groupby('calificacion')['precio'].agg(['count', 'mean']).reset_index()
promedios_por_rating.columns = ['calificacion_estrellas', 'cantidad_libros', 'precio_promedio']
promedios_por_rating

## 4. Extracción Directa de Tablas HTML con `pd.read_html()`

Pandas incluye internamente la función `pd.read_html()` que busca automáticamente etiquetas `<table>` en una página web o cadena HTML y las transforma directamente a DataFrames sin requerir selectores manuales.

In [ ]:
# Ejemplo de tabla HTML con indicadores
tabla_html = '''
<table border="1" class="dataframe">
  <thead>
    <tr>
      <th>Categoria</th>
      <th>Indice_Crecimiento_%</th>
      <th>Ventas_Esperadas_USD</th>
      <th>Riesgo</th>
    </tr>
  </thead>
  <tbody>
    <tr><td>Tecnología</td><td>14.5</td><td>450000</td><td>Moderado</td></tr>
    <tr><td>Papelería</td><td>3.2</td><td>120000</td><td>Bajo</td></tr>
    <tr><td>Alimentos</td><td>8.9</td><td>680000</td><td>Bajo</td></tr>
    <tr><td>Hogar</td><td>6.1</td><td>230000</td><td>Moderado</td></tr>
    <tr><td>Ropa</td><td>11.4</td><td>390000</td><td>Alto</td></tr>
  </tbody>
</table>
'''

# Lectura directa con pandas
tablas = pd.read_html(tabla_html)
df_tabla = tablas[0]
print("Tabla HTML extraída directamente con pd.read_html():")
df_tabla

## 5. Exportación de Datos Obtenidos a CSV

In [ ]:
# Guardar los DataFrames resultantes del web scraping a la carpeta data
df_libros.to_csv('data/scraped_libros.csv', index=False)
df_tabla.to_csv('data/scraped_indicadores_mercado.csv', index=False)

print("Archivos exportados exitosamente en 'data/':")
print("  - data/scraped_libros.csv")
print("  - data/scraped_indicadores_mercado.csv")